In [0]:
%python
file_path = "/Volumes/workspace/genomics/raw_data/GSE236379_Gene_Expression_Count.txt.gz"

raw = spark.read.text(file_path)

raw.show(10, truncate=False)

In [0]:
%python
print("number of lines: ", raw.count())

In [0]:
%python
from pyspark.sql import functions as F

counts = (
    spark.read
    .option("header", "true")
    .option("sep", "\t")
    .option("inferSchema", "true")
    .csv(file_path)
)

counts.show(5)

In [0]:
%python
from pyspark.sql import functions as F

raw = spark.read.text(file_path)

# Split every line by tab
split_data = raw.select(
    F.split(F.col("value"), "\t").alias("fields")
)

# Remove the header
data_rows = split_data.filter(
    F.col("fields")[0] != "d20_Hyp1_S4_R1_001"
)

# Create gene + 12 sample columns
counts = data_rows.select(
    F.col("fields")[0].alias("gene"),
    *[
        F.col("fields")[i].cast("long").alias(name)
        for i, name in enumerate(
            [
                "d20_Hyp1_S4_R1_001",
                "d20_Hyp2_S5_R1_001",
                "d20_Hyp3_S6_R1_001",
                "d20_Norm1_S1_R1_001",
                "d20_Norm2_S2_R1_001",
                "d20_Norm3_S3_R1_001",
                "d25_Hyp1_S10_R1_001",
                "d25_Hyp2_S11_R1_001",
                "d25_Hyp3_S12_R1_001",
                "d25_Norm1_S7_R1_001",
                "d25_Norm2_S8_R1_001",
                "d25_Norm3_S9_R1_001"
            ],
            start=1
        )
    ]
)

counts.show(5, truncate=False)

In [0]:
%python
print("Rows:", counts.count())
print("Columns:", len(counts.columns))

counts.printSchema()

In [0]:
%python
from pyspark.sql import functions as F

missing = counts.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in counts.columns
    ]
)

missing.show()

In [0]:
%python
counts = data_rows.select(
    F.col("fields")[0].cast("string").alias("gene"),
    *[
        F.col("fields")[i].cast("long").alias(name)
        for i, name in enumerate(
            [
                "d20_Hyp1_S4_R1_001",
                "d20_Hyp2_S5_R1_001",
                "d20_Hyp3_S6_R1_001",
                "d20_Norm1_S1_R1_001",
                "d20_Norm2_S2_R1_001",
                "d20_Norm3_S3_R1_001",
                "d25_Hyp1_S10_R1_001",
                "d25_Hyp2_S11_R1_001",
                "d25_Hyp3_S12_R1_001",
                "d25_Norm1_S7_R1_001",
                "d25_Norm2_S8_R1_001",
                "d25_Norm3_S9_R1_001"
            ],
            start=1
        )
    ]
)

counts.printSchema()
counts.show(5, truncate=False)

In [0]:
%python
from pyspark.sql import functions as F

duplicate_genes = (
    counts
    .groupBy("gene")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
)

duplicate_genes.show(20, truncate=False)

In [0]:
%python
from pyspark.sql import functions as F

# ============================================================
# 1. Read the complete raw file
# ============================================================

file_path = "/Volumes/workspace/genomics/raw_data/GSE236379_Gene_Expression_Count.txt.gz"

raw = spark.read.text(file_path)


# ============================================================
# 2. Split each line by TAB
# ============================================================

split_data = (
    raw
    .select(F.split(F.col("value"), "\t").alias("fields"))
)


# ============================================================
# 3. Remove the header
# ============================================================

data_rows = (
    split_data
    .filter(F.col("fields")[0] != "d20_Hyp1_S4_R1_001")
)


# ============================================================
# 4. Define sample names
# ============================================================

sample_names = [
    "d20_Hyp1_S4_R1_001",
    "d20_Hyp2_S5_R1_001",
    "d20_Hyp3_S6_R1_001",
    "d20_Norm1_S1_R1_001",
    "d20_Norm2_S2_R1_001",
    "d20_Norm3_S3_R1_001",
    "d25_Hyp1_S10_R1_001",
    "d25_Hyp2_S11_R1_001",
    "d25_Hyp3_S12_R1_001",
    "d25_Norm1_S7_R1_001",
    "d25_Norm2_S8_R1_001",
    "d25_Norm3_S9_R1_001"
]


# ============================================================
# 5. Create structured expression DataFrame
# ============================================================

counts = data_rows.select(
    F.trim(F.col("fields")[0]).alias("gene"),
    *[
        F.col("fields")[i]
        .cast("long")
        .alias(sample)
        for i, sample in enumerate(sample_names, start=1)
    ]
)


# ============================================================
# 6. Remove invalid gene names
# ============================================================

counts = (
    counts
    .filter(
        F.col("gene").isNotNull() &
        (F.trim(F.col("gene")) != "")
    )
)


# ============================================================
# 7. Remove duplicate genes
# ============================================================

counts = (
    counts
    .dropDuplicates(["gene"])
)


# ============================================================
# 8. Replace NULL expression counts with 0
# ============================================================

counts = counts.fillna(0, subset=sample_names)


# ============================================================
# 9. Make sure expression counts are not negative
# ============================================================

negative_counts = None

for sample in sample_names:

    condition = F.col(sample) < 0

    if negative_counts is None:
        negative_counts = condition
    else:
        negative_counts = negative_counts | condition

counts = counts.filter(~negative_counts)


# ============================================================
# 10. Convert wide format → long format
# ============================================================

stack_expression = ", ".join(
    [f"'{sample}', `{sample}`" for sample in sample_names]
)

expression_long = counts.select(
    "gene",
    F.expr(
        f"stack({len(sample_names)}, {stack_expression}) "
        "as (sample, count)"
    )
)


# ============================================================
# 11. Create sample metadata
# ============================================================

metadata = spark.createDataFrame(
    [
        ("d20_Hyp1_S4_R1_001", 20, "Hypoxia", 1),
        ("d20_Hyp2_S5_R1_001", 20, "Hypoxia", 2),
        ("d20_Hyp3_S6_R1_001", 20, "Hypoxia", 3),

        ("d20_Norm1_S1_R1_001", 20, "Normal", 1),
        ("d20_Norm2_S2_R1_001", 20, "Normal", 2),
        ("d20_Norm3_S3_R1_001", 20, "Normal", 3),

        ("d25_Hyp1_S10_R1_001", 25, "Hypoxia", 1),
        ("d25_Hyp2_S11_R1_001", 25, "Hypoxia", 2),
        ("d25_Hyp3_S12_R1_001", 25, "Hypoxia", 3),

        ("d25_Norm1_S7_R1_001", 25, "Normal", 1),
        ("d25_Norm2_S8_R1_001", 25, "Normal", 2),
        ("d25_Norm3_S9_R1_001", 25, "Normal", 3)
    ],
    ["sample", "day", "condition", "replicate"]
)


# ============================================================
# 12. Join expression data with biological metadata
# ============================================================

processed_data = (
    expression_long
    .join(
        metadata,
        on="sample",
        how="inner"
    )
)


# ============================================================
# 13. Arrange columns
# ============================================================

processed_data = processed_data.select(
    "gene",
    "sample",
    "day",
    "condition",
    "replicate",
    "count"
)


# ============================================================
# 14. Cache because we'll reuse this DataFrame
# ============================================================

processed_data.cache()


# ============================================================
# 15. Display the processed dataset
# ============================================================

processed_data.show(20, truncate=False)


# ============================================================
# 16. Basic statistics
# ============================================================

print("Genes:", counts.count())
print("Samples:", metadata.count())
print("Processed rows:", processed_data.count())

In [0]:
%python
processed_data.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.genomics.silver_expression")

In [0]:
%python
spark.sql("""
DESCRIBE TABLE workspace.genomics.silver_expression
""").show(truncate=False)

In [0]:
%python
spark.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT gene) AS genes,
    COUNT(DISTINCT sample) AS samples
FROM workspace.genomics.silver_expression
""").show()

In [0]:
%python
spark.sql("""
SELECT
    day,
    condition,
    COUNT(DISTINCT sample) AS sample_count
FROM workspace.genomics.silver_expression
GROUP BY day, condition
ORDER BY day, condition
""").show()

In [0]:
%python
spark.sql("""
SELECT
    gene,
    day,
    condition,
    ROUND(AVG(count), 2) AS avg_expression
FROM workspace.genomics.silver_expression
GROUP BY gene, day, condition
ORDER BY gene, day, condition
""").show(20, truncate=False)

In [0]:
%python
spark.sql("""
WITH gene_averages AS (
    SELECT
        gene,
        day,
        condition,
        AVG(count) AS avg_expression
    FROM workspace.genomics.silver_expression
    GROUP BY gene, day, condition
)

SELECT
    gene,
    day,
    ROUND(
        MAX(CASE WHEN condition = 'Hypoxia'
                 THEN avg_expression END), 2
    ) AS hypoxia_avg,

    ROUND(
        MAX(CASE WHEN condition = 'Normal'
                 THEN avg_expression END), 2
    ) AS normal_avg,

    ROUND(
        MAX(CASE WHEN condition = 'Hypoxia'
                 THEN avg_expression END)
        -
        MAX(CASE WHEN condition = 'Normal'
                 THEN avg_expression END), 2
    ) AS expression_difference

FROM gene_averages
GROUP BY gene, day
ORDER BY expression_difference DESC
""").show(20, truncate=False)

In [0]:
%python
spark.sql("""
WITH gene_averages AS (
    SELECT
        gene,
        day,
        condition,
        AVG(count) AS avg_expression
    FROM workspace.genomics.silver_expression
    GROUP BY gene, day, condition
)

SELECT
    gene,
    day,

    ROUND(
        MAX(CASE WHEN condition = 'Hypoxia'
                 THEN avg_expression END), 2
    ) AS hypoxia_avg,

    ROUND(
        MAX(CASE WHEN condition = 'Normal'
                 THEN avg_expression END), 2
    ) AS normal_avg,

    ROUND(
        MAX(CASE WHEN condition = 'Hypoxia'
                 THEN avg_expression END)
        -
        MAX(CASE WHEN condition = 'Normal'
                 THEN avg_expression END), 2
    ) AS expression_difference

FROM gene_averages
GROUP BY gene, day
ORDER BY expression_difference DESC
""").show(20, truncate=False)

In [0]:
%python
spark.sql("""
WITH gene_averages AS (
    SELECT
        gene,
        day,
        condition,
        AVG(count) AS avg_expression
    FROM workspace.genomics.silver_expression
    GROUP BY gene, day, condition
),

comparison AS (
    SELECT
        gene,
        day,

        MAX(
            CASE
                WHEN condition = 'Hypoxia'
                THEN avg_expression
            END
        ) AS hypoxia_avg,

        MAX(
            CASE
                WHEN condition = 'Normal'
                THEN avg_expression
            END
        ) AS normal_avg

    FROM gene_averages
    GROUP BY gene, day
)

SELECT
    gene,
    day,
    ROUND(hypoxia_avg, 2) AS hypoxia_avg,
    ROUND(normal_avg, 2) AS normal_avg,

    ROUND(
        hypoxia_avg / NULLIF(normal_avg, 0),
        3
    ) AS fold_change

FROM comparison
WHERE normal_avg > 0
ORDER BY fold_change DESC
""").show(20, truncate=False)

In [0]:
%sql
SELECT
    gene,
    day,

    ROUND(
        AVG(CASE WHEN condition = 'Hypoxia' THEN count END),
        2
    ) AS hypoxia_avg,

    ROUND(
        AVG(CASE WHEN condition = 'Normal' THEN count END),
        2
    ) AS normal_avg,

    ROUND(
        LOG2(
            AVG(CASE WHEN condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS log2_fold_change

FROM workspace.genomics.silver_expression

GROUP BY gene, day

HAVING
    AVG(CASE WHEN condition = 'Normal' THEN count END) > 0

ORDER BY log2_fold_change DESC

In [0]:
%sql
SELECT
    gene,
    day,

    ROUND(
        AVG(CASE WHEN condition = 'Hypoxia' THEN count END),
        2
    ) AS hypoxia_avg,

    ROUND(
        AVG(CASE WHEN condition = 'Normal' THEN count END),
        2
    ) AS normal_avg,

    ROUND(
        LOG2(
            AVG(CASE WHEN condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS log2_fold_change

FROM workspace.genomics.silver_expression

GROUP BY gene, day

HAVING
    AVG(CASE WHEN condition = 'Normal' THEN count END) >= 10

ORDER BY log2_fold_change DESC

In [0]:
%sql
SELECT
    MAX(log2_fold_change) AS max_log2_fold_change
FROM (
    SELECT
        gene,
        day,
        LOG2(
            AVG(CASE WHEN condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN condition = 'Normal' THEN count END),
                0
            )
        ) AS log2_fold_change
    FROM workspace.genomics.silver_expression
    GROUP BY gene, day
    HAVING AVG(CASE WHEN condition = 'Normal' THEN count END) >= 10
) t;

In [0]:
%sql
SELECT COUNT(*) AS candidate_genes
FROM workspace.genomics.gold_hypoxia_candidates;

In [0]:
%sql
SELECT
    gene,
    day,
    hypoxia_avg,
    normal_avg,
    log2_fold_change
FROM workspace.genomics.gold_hypoxia_candidates
ORDER BY log2_fold_change DESC
LIMIT 20;

In [0]:
%sql
SELECT
    gene,

    ROUND(
        MAX(CASE WHEN day = 20 THEN log2_fold_change END),
        3
    ) AS day20_log2fc,

    ROUND(
        MAX(CASE WHEN day = 25 THEN log2_fold_change END),
        3
    ) AS day25_log2fc

FROM workspace.genomics.gold_hypoxia_candidates

GROUP BY gene

HAVING
    MAX(CASE WHEN day = 20 THEN log2_fold_change END) IS NOT NULL
    AND
    MAX(CASE WHEN day = 25 THEN log2_fold_change END) IS NOT NULL

ORDER BY
    day20_log2fc DESC;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.genomics.gold_day_comparison AS

SELECT
    gene,

    ROUND(
        MAX(CASE WHEN day = 20 THEN log2_fold_change END),
        3
    ) AS day20_log2fc,

    ROUND(
        MAX(CASE WHEN day = 25 THEN log2_fold_change END),
        3
    ) AS day25_log2fc,

    ROUND(
        MAX(CASE WHEN day = 25 THEN log2_fold_change END)
        -
        MAX(CASE WHEN day = 20 THEN log2_fold_change END),
        3
    ) AS change_day25_vs_day20

FROM workspace.genomics.gold_hypoxia_candidates

GROUP BY gene

HAVING
    MAX(CASE WHEN day = 20 THEN log2_fold_change END) IS NOT NULL
    AND
    MAX(CASE WHEN day = 25 THEN log2_fold_change END) IS NOT NULL;

In [0]:
%sql
SELECT
    gene,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 20 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day20_log2fc,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 25 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day25_log2fc

FROM workspace.genomics.silver_expression

GROUP BY gene

HAVING
    AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END) >= 10
    AND
    AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END) >= 10

ORDER BY day20_log2fc DESC

LIMIT 20;

In [0]:
%sql
SELECT
    gene,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 20 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day20_log2fc,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 25 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day25_log2fc,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 25 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END),
                0
            )
        )
        -
        LOG2(
            AVG(CASE WHEN day = 20 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS response_change

FROM workspace.genomics.silver_expression

GROUP BY gene

HAVING
    AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END) >= 10
    AND
    AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END) >= 10

ORDER BY response_change DESC

LIMIT 20;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.genomics.gold_time_response AS

SELECT
    gene,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 20 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day20_log2fc,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 25 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS day25_log2fc,

    ROUND(
        LOG2(
            AVG(CASE WHEN day = 25 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END),
                0
            )
        )
        -
        LOG2(
            AVG(CASE WHEN day = 20 AND condition = 'Hypoxia' THEN count END)
            /
            NULLIF(
                AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END),
                0
            )
        ),
        3
    ) AS response_change

FROM workspace.genomics.silver_expression

GROUP BY gene

HAVING
    AVG(CASE WHEN day = 20 AND condition = 'Normal' THEN count END) >= 10
    AND
    AVG(CASE WHEN day = 25 AND condition = 'Normal' THEN count END) >= 10;

In [0]:
%sql
SELECT COUNT(*) AS total_genes
FROM workspace.genomics.gold_time_response;

In [0]:
%sql
SELECT
    gene,
    day20_log2fc,
    day25_log2fc,
    response_change
FROM workspace.genomics.gold_time_response
WHERE response_change >= 2
ORDER BY response_change DESC
LIMIT 20;

In [0]:
%sql
SELECT
    gene,
    day20_log2fc,
    day25_log2fc,
    response_change,

    CASE
        WHEN day20_log2fc <= 0
             AND day25_log2fc >= 1
            THEN 'Late induced'

        WHEN day20_log2fc >= 1
             AND day25_log2fc < day20_log2fc
            THEN 'Early induced'

        WHEN day20_log2fc >= 1
             AND day25_log2fc >= 1
            THEN 'Consistently induced'

        ELSE 'Other'
    END AS response_pattern

FROM workspace.genomics.gold_time_response;

In [0]:
%sql

WITH classified AS (
    SELECT
        gene,
        CASE
            WHEN day20_log2fc <= 0
                 AND day25_log2fc >= 1
                THEN 'Late induced'

            WHEN day20_log2fc >= 1
                 AND day25_log2fc < day20_log2fc
                THEN 'Early induced'

            WHEN day20_log2fc >= 1
                 AND day25_log2fc >= 1
                THEN 'Consistently induced'

            ELSE 'Other'
        END AS response_pattern

    FROM workspace.genomics.gold_time_response
)

SELECT
    response_pattern,
    COUNT(*) AS gene_count
FROM classified
GROUP BY response_pattern
ORDER BY gene_count DESC;

In [0]:
SELECT
    gene,
    day20_log2fc,
    day25_log2fc
FROM workspace.genomics.gold_time_response
WHERE day20_log2fc <= 0
  AND day25_log2fc >= 1
ORDER BY day25_log2fc DESC
LIMIT 20;

In [0]:
DESCRIBE workspace.genomics.silver_expression;

In [0]:
SELECT
    gene,
    day,
    condition,
    ROUND(AVG(count), 2) AS avg_expression
FROM workspace.genomics.silver_expression
WHERE gene IN ('MMP7', 'SPINK1', 'S100A9',
               'S100A8', 'A2M', 'SPP1')
GROUP BY gene, day, condition
ORDER BY gene, day, condition;

In [0]:
SELECT
    day,
    condition,
    ROUND(AVG(count), 2) AS avg_expression
FROM workspace.genomics.silver_expression
WHERE gene = 'MMP7'
GROUP BY day, condition
ORDER BY day, condition;